# 01 - Setup and smoke test

Confirms the environment, loads the base model, and runs a single night end to
end so that a longer run does not fail on something trivial three hours in.

Expect roughly 10-15 minutes on a T4.

## Environment

In [ ]:
import os, sys, subprocess

REPO = "/kaggle/working/skill-library"
if not os.path.exists(REPO):
    # On Kaggle, add the repo as a dataset or clone it here.
    REPO = os.getcwd()
sys.path.insert(0, REPO)
os.chdir(REPO)

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "peft>=0.11", "transformers>=4.40", "accelerate"], check=False)

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"{name}  {total:.1f} GB  bf16={torch.cuda.is_bf16_supported()}")

## The crafting environment

CPU-only, and independent of the model.

In [ ]:
from env import curriculum_order, validate_curriculum
from env.reference import verify_all

validate_curriculum()
steps = verify_all()
print(f"{len(steps)} tasks, all reference programs verified")
for task in curriculum_order():
    print(f"  tier {task.tier}  {task.name:<22} -> {task.goal_text()}")

## Base model

In [ ]:
from baselines.gpu_runner import load_config
from sleep.lora import load_base_model

config = load_config("configs/default.yaml")
model, tokenizer = load_base_model(
    config["model"]["name"], config["model"]["device"], config["model"]["dtype"]
)
print(f"loaded {config['model']['name']}")
print(f"{sum(p.numel() for p in model.parameters())/1e9:.2f}B parameters")

## What the model does before any consolidation

The base model has neither the library nor an adapter here, so weak output is
the expected result. This is the starting point the rest of the project moves
away from.

In [ ]:
# What does the base model do with no library and no adapter?
from env.tasks import TASKS
from library.retrieval import RetrievalConfig, build_prompt
from library.store import SkillLibrary

task = TASKS["craft_wood_pickaxe"]
prompt = build_prompt(SkillLibrary(), task, RetrievalConfig(), "at=forest | inventory: empty")
print(prompt)
print("-" * 60)

from sleep.lora import generate_with
print(generate_with(model, tokenizer, prompt, config["model"]["device"], 96))

## One night

In [ ]:
# One night end to end: learn three skills, consolidate, checkpoint.
from baselines.gpu_runner import GPURun

config["run"]["run_id"] = "gpu_smoke"
run = GPURun(config, method="ours", model=model, tokenizer=tokenizer)
loop = run.run(max_nights=1)

print(f"library: {len(run.library)} skills -> {run.library.names}")
for night in run.consolidator.nights:
    s = night.stats
    print(f"night {night.night}: {s.n_new} new + {s.n_replay} replayed, "
          f"{s.steps} steps, loss {s.loss:.3f}, {s.seconds:.0f}s")
print(f"adapter: {run.backend.stored_parameters():,} parameters")

## Recall from weights

In [ ]:
# Did the night put anything in the weights? Decode with no library in context.
for skill in run.library:
    score = run.backend.knows(skill.name)
    print(f"{skill.name:<22} recall {score:.2f}")

## Resume

In [ ]:
# The resume path, which every later session depends on.
resumed = GPURun(config, method="ours", model=model, tokenizer=tokenizer)
night = resumed.resume()
print(f"resumed at night {night} with {len(resumed.library)} skills")
assert resumed.library.names == run.library.names
print("checkpoint round-trips")